In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Thinking levels and multi-model orchestration with Gemini 4

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fthinking%2Fgemini_4_thinking_and_orchestration.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/thinking/gemini_4_thinking_and_orchestration.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author(s) |
| --- |
| [Eric Dong](https://github.com/gericdong) |
| [Holt Skinner](https://github.com/holtskinner) |

## Overview

Gemini 4 Carbon, Neon, and Hydrogen share the same request schema, 1M-token context window, and tool support across the `GenerateContent` and `Interactions` APIs, so you can switch models per turn without changing your request code (Argon shares Carbon's `LOW`–`HIGH` thinking range and `MEDIUM` default). They differ in default `thinking_level`, supported thinking levels, and price (see [Model Garden pricing](https://cloud.google.com/model-garden/pricing) and the [Gemini 4 Neon guide](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/guides/gemini-4-neon)):

| Model | Model ID | Thinking Levels | Default | Per-Request Output Cap | Price (per 1M In / Out) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Gemini 4 Hydrogen** | `gemini-4-hydrogen` | `LOW`, `MEDIUM`, `HIGH`, `XHIGH` | `HIGH` | 64K streaming & non-streaming | \$1.50 / \$7.50 (\$0.075 cached) |
| **Gemini 4 Neon** | `gemini-4-neon` | `LOW`, `MEDIUM`, `HIGH`, `XHIGH` | `HIGH` | 64K streaming & non-streaming | \$3.00 / \$15.00 (\$0.15 cached) |
| **Gemini 4 Carbon** | `gemini-4-carbon` | `LOW`, `MEDIUM`, `HIGH` | `MEDIUM` | 64K streaming / 32K non-streaming | \$4.00 / \$20.00 (\$0.20 cached) |

This notebook covers four patterns:
- Comparing default thinking levels and thought summaries (`include_thoughts=True`) across Hydrogen, Neon, and Carbon.
- Comparing `LOW`, `MEDIUM`, `HIGH`, and `XHIGH` on Neon, and capping `XHIGH` to `HIGH` for Carbon.
- Switching between Neon and Hydrogen across turns on the `Interactions API` via `previous_interaction_id`.
- Preserving `thought_signature` parts on parallel tool calls and reusing per-model caches on `GenerateContent`.

## Get started

### Install Google Gen AI SDK

Use `google-genai >= 2.34.0` for `XHIGH` thinking levels and `Interactions API` support.

> **Note:** You can safely ignore `pip` dependency resolver errors from unrelated pre-installed packages in Colab.

In [ ]:
%pip install --upgrade --quiet google-genai

### Authenticate your notebook environment

Authenticate your account if running in Google Colab.

In [2]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [3]:
import os
import time
from typing import Any

from google import genai
from google.genai import types
from google.genai.errors import APIError
from IPython.display import Markdown, display

### Set Google Cloud project information

Set your Google Cloud project ID and initialize `genai.Client` for Gemini for business.

In [4]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
LOCATION = "global"  # @param ["global", "us", "eu"]

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"

client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(
        timeout=240000,
        retry_options=types.HttpRetryOptions(attempts=3),
    ),
)

### Define model IDs

In [5]:
HYDROGEN_MODEL = "gemini-4-hydrogen"
NEON_MODEL = "gemini-4-neon"
CARBON_MODEL = "gemini-4-carbon"

## Thinking levels across Gemini 4 models

### Default thinking behavior and thought summaries

If you omit `thinking_level`, Carbon defaults to `MEDIUM` while Neon and Hydrogen default to `HIGH`. Setting `include_thoughts=True` returns short summaries of the model's reasoning alongside `usage_metadata.thoughts_token_count`.

On Gemini 4, passing legacy `thinking_budget` or `thinking_level="MINIMAL"` returns `HTTP 400`, and custom `temperature`, `top_p`, and `top_k` values are ignored (control behavior with `thinking_level` and `response_schema`).

In [6]:
def generate_with_metrics(
    model: str,
    prompt: str,
    thinking_level: str | None = None,
) -> dict[str, Any]:
    thinking_kwargs: dict[str, Any] = {"include_thoughts": True}
    if thinking_level is not None:
        thinking_kwargs["thinking_level"] = thinking_level

    start = time.perf_counter()
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(**thinking_kwargs),
        ),
    )
    elapsed_s = time.perf_counter() - start
    parts = response.candidates[0].content.parts
    thoughts = "".join(p.text for p in parts if p.thought and p.text)
    answer = "".join(p.text for p in parts if not p.thought and p.text)
    usage = response.usage_metadata
    return {
        "model": model,
        "elapsed_s": elapsed_s,
        "thought_tokens": usage.thoughts_token_count or 0,
        "output_tokens": usage.candidates_token_count or 0,
        "thoughts": thoughts.strip(),
        "answer": answer.strip(),
    }

Run the same cache-invalidation prompt across all three models at their default settings.

In [7]:
CONCURRENCY_PROMPT = (
    "Service A deletes Redis key `user:42` BEFORE committing its SQL UPDATE "
    "under READ COMMITTED, while Service B handles a cache miss on `user:42`. "
    "Trace the exact interleaving that caches stale data permanently, and give "
    "a 3-bullet fix."
)

default_runs = [
    generate_with_metrics(model_id, CONCURRENCY_PROMPT)
    for model_id in (HYDROGEN_MODEL, NEON_MODEL, CARBON_MODEL)
]

header = (
    f"{'Model':<20} | {'Default':<8} | {'Latency':>8} | "
    f"{'Thought Tok':>11} | {'Out Tok':>7}"
)
print(header)
print("-" * len(header))
for run in default_runs:
    default_lvl = "MEDIUM" if run["model"] == CARBON_MODEL else "HIGH"
    print(
        f"{run['model']:<20} | {default_lvl:<8} | "
        f"{run['elapsed_s']:>7.2f}s | {run['thought_tokens']:>11} | "
        f"{run['output_tokens']:>7}"
    )

print("\nSample thought summary (gemini-4-neon):")
summary = default_runs[1]["thoughts"]
print(summary[:400].rsplit(" ", 1)[0] + "..." if len(summary) > 400 else summary)

Model                | Default  |  Latency | Thought Tok | Out Tok
------------------------------------------------------------------
gemini-4-hydrogen    | HIGH     |   14.62s |        1246 |     532
gemini-4-neon        | HIGH     |   23.73s |        1494 |     479
gemini-4-carbon      | MEDIUM   |   84.05s |         905 |     586

Sample thought summary (gemini-4-neon):
**My Interleaved Thought Process: The Stale Cache Conundrum**

Alright, I've been wrestling with a particularly thorny race condition, and I think I've finally pinned down the exact sequence of events that leads to *permanently* cached stale data, even under `READ COMMITTED` isolation. It's a classic cache-aside pattern gone wrong, specifically when the cache invalidation happens *before* the...


### Compare LOW, MEDIUM, HIGH, and XHIGH on Neon

On Neon and Hydrogen, `HIGH` is the default for coding and agent loops, `MEDIUM` and `LOW` cut latency on simpler tool or routing turns, and `XHIGH` spends extra thinking tokens on hard math, science, or debugging.

In [8]:
REASONING_PROMPT = (
    "Find the smallest integer n > 1 such that n^3 + 2n + 3 is divisible by "
    "49, and verify the quotient in 2 sentences."
)

print(f"{'Level':<8} | {'Latency':>8} | {'Thought Tokens':>14} | {'Out Tok':>9}")
print("-" * 50)
for level in ("LOW", "MEDIUM", "HIGH", "XHIGH"):
    metrics = generate_with_metrics(
        NEON_MODEL, REASONING_PROMPT, thinking_level=level
    )
    print(
        f"{level:<8} | {metrics['elapsed_s']:>7.2f}s | "
        f"{metrics['thought_tokens']:>14} | {metrics['output_tokens']:>9}"
    )

Level    |  Latency | Thought Tokens |   Out Tok
--------------------------------------------------
LOW      |   17.02s |           1594 |       213
MEDIUM   |   19.44s |           2093 |       197
HIGH     |   22.59s |           2518 |       210
XHIGH    |   93.92s |          10665 |       254


### Cap XHIGH to HIGH for Carbon

Two details to handle when switching between models:
- Carbon (and Argon) stop at `HIGH` and return `HTTP 400` if you pass `XHIGH`.
- `GenerateContent` accepts uppercase or lowercase `thinking_level` strings in the Python SDK (uppercase in REST), while the `Interactions API` requires lowercase (`"high"`) and returns `HTTP 400` on uppercase.

In [9]:
VALID_LEVELS = {"LOW", "MEDIUM", "HIGH", "XHIGH"}
HIGH_MAX_MODELS = {CARBON_MODEL, "gemini-4-argon"}


def get_thinking_level(
    model: str,
    requested_level: str,
    *,
    for_interactions: bool = False,
) -> str:
    level = requested_level.strip().upper()
    if level not in VALID_LEVELS:
        raise ValueError(f"Unsupported thinking level: {requested_level}")
    if model in HIGH_MAX_MODELS and level == "XHIGH":
        level = "HIGH"
    return level.lower() if for_interactions else level

Test passing `XHIGH` directly to Carbon (`HTTP 400`) vs. capping it to `HIGH`.

In [10]:
try:
    client.models.generate_content(
        model=CARBON_MODEL,
        contents="Return OK.",
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="XHIGH"),
        ),
    )
except APIError as err:
    print(f"Expected error on XHIGH -> HTTP {err.code}: {err.message[:75]}")

safe_level = get_thinking_level(CARBON_MODEL, "XHIGH")
ok_response = client.models.generate_content(
    model=CARBON_MODEL,
    contents="Return OK.",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level=safe_level),
    ),
)
print(
    f"Capped level for {CARBON_MODEL}: {safe_level} -> "
    f"{ok_response.text.strip()}"
)

Expected error on XHIGH -> HTTP 400: Thinking level is unsupported: THINKING_LEVEL_XHIGH
Capped level for gemini-4-carbon: HIGH -> OK


## Switch models across turns with the Interactions API

With `store=True`, the `Interactions API` keeps conversation state on the server for 7 days. You can switch `model` and `thinking_level` on any turn by passing `previous_interaction_id` instead of resending message history. `generation_config` and `system_instruction` apply only to the current turn, so pass them on each call.

A practical pattern used in tools like [Antigravity Orchestrator](https://antigravity.google/docs/models#orchestrator) pairs a frontier **Planner** (Neon or Carbon) with a fast **Executor** (Hydrogen) over shared conversation state:
- **Plan:** The Planner explores context, writes a concrete plan with re-plan triggers, or answers simple questions directly without handing off.
- **Execute:** Hydrogen writes code, applies edits, and runs tests.
- **Escalate & review:** If a verification step fails twice or hits an unsettled design choice, control returns to the Planner to repair the plan; once done, the Planner reviews the final work.

Here Neon (`high`) writes a short spec, Hydrogen (`high` and `medium`) implements the class and `pytest` suite, and Neon (`xhigh`) reviews the result for security bugs.

### Step 1: Write the spec with Neon

In [11]:
stored_interaction_ids: list[str] = []

plan_turn = client.interactions.create(
    model=NEON_MODEL,
    system_instruction="Keep specifications concise and actionable.",
    input=(
        "Design a 3-step specification for a Python `WebhookVerifier` class "
        "that validates `X-Signature-256` HMAC-SHA256 headers, enforces a "
        "300-second timestamp window, and blocks replayed event IDs."
    ),
    generation_config={
        "thinking_level": get_thinking_level(
            NEON_MODEL, "HIGH", for_interactions=True
        ),
    },
    store=True,
    timeout=180,
)
stored_interaction_ids.append(plan_turn.id)

print(
    f"Turn 1 [{NEON_MODEL} high] "
    f"thought_tokens={plan_turn.usage.total_thought_tokens}, "
    f"output_tokens={plan_turn.usage.total_output_tokens}"
)
spec_lines = plan_turn.output_text.splitlines()
headings = [ln for ln in spec_lines if ln.startswith("#")]
print(f"Generated {len(spec_lines)} lines:")
print("\n".join(headings))

Turn 1 [gemini-4-neon high] thought_tokens=1597, output_tokens=805
Generated 48 lines:
### Step 1: Class Interface & Exceptions
### Step 2: Verification Pipeline (`verify`)
### Step 3: Replay Store Protocol & Implementation


### Step 2: Implement code and tests with Hydrogen

Pass `previous_interaction_id=plan_turn.id` to Hydrogen.

In [12]:
impl_turn = client.interactions.create(
    model=HYDROGEN_MODEL,
    system_instruction=(
        "Make minimal, self-contained edits using Python standard library only."
    ),
    input=(
        "Implement `WebhookVerifier` in a single Python code block following "
        "the 3-step spec using descriptive variable names."
    ),
    previous_interaction_id=plan_turn.id,
    generation_config={
        "thinking_level": get_thinking_level(
            HYDROGEN_MODEL, "HIGH", for_interactions=True
        ),
    },
    store=True,
    timeout=240,
)
stored_interaction_ids.append(impl_turn.id)

print(
    f"Turn 2 [{HYDROGEN_MODEL}] "
    f"thought_tokens={impl_turn.usage.total_thought_tokens}, "
    f"output_tokens={impl_turn.usage.total_output_tokens}"
)
impl_lines = impl_turn.output_text.splitlines()
defs = [ln.strip() for ln in impl_lines if ln.lstrip().startswith(("class ", "def "))]
print(f"Generated {len(impl_lines)} lines ({len(defs)} classes/methods):")
print("\n".join(f"  {d}" for d in defs[:10]))

Turn 2 [gemini-4-hydrogen] thought_tokens=1750, output_tokens=1290
Generated 142 lines (12 classes/methods):
  class WebhookVerificationError(Exception):
  class InvalidSignatureError(WebhookVerificationError):
  class ExpiredTimestampError(WebhookVerificationError):
  class ReplayDetectedError(WebhookVerificationError):
  class ReplayStore(Protocol):
  def claim(self, event_id: str, ttl: int) -> bool:
  class InMemoryReplayStore:
  def __init__(self) -> None:
  def claim(self, event_id: str, ttl: int) -> bool:
  class WebhookVerifier:


Stream a second Hydrogen turn at `medium` to generate unit tests, and grab the completed `interaction` ID from the `interaction.completed` event.

In [13]:
test_turn = None
streamed_text: list[str] = []

for event in client.interactions.create(
    model=HYDROGEN_MODEL,
    input=(
        "Write 4 concise `pytest` tests covering valid, bad signature, "
        "expired timestamp, and replay cases using plain test strings such as 'test_secret'."
    ),
    previous_interaction_id=impl_turn.id,
    generation_config={
        "thinking_level": get_thinking_level(
            HYDROGEN_MODEL, "MEDIUM", for_interactions=True
        ),
    },
    store=True,
    stream=True,
    timeout=240,
):
    if event.event_type == "step.delta" and event.delta.type == "text":
        streamed_text.append(event.delta.text)
    elif (
        event.event_type == "interaction.completed"
        and event.interaction.status != "requires_continuation"
    ):
        test_turn = event.interaction

stored_interaction_ids.append(test_turn.id)
print(
    f"Turn 3 [{HYDROGEN_MODEL} stream] "
    f"thought_tokens={test_turn.usage.total_thought_tokens}, "
    f"output_tokens={test_turn.usage.total_output_tokens}"
)
test_code = "".join(streamed_text)
test_lines = test_code.splitlines()
tests = [ln.strip() for ln in test_lines if ln.lstrip().startswith("def test_")]
print(f"Generated {len(test_lines)} lines ({len(tests)} tests):")
print("\n".join(f"  {t}" for t in tests))

Turn 3 [gemini-4-hydrogen stream] thought_tokens=453, output_tokens=612
Generated 68 lines (4 tests):
  def test_verify_valid_webhook() -> None:
  def test_verify_bad_signature_raises() -> None:
  def test_verify_expired_timestamp_raises() -> None:
  def test_verify_replayed_event_id_raises() -> None:


### Step 3: Review with Neon at XHIGH

Pass `previous_interaction_id=test_turn.id` back to Neon at `xhigh`.

In [14]:
verify_turn = None
verify_chunks: list[str] = []

for event in client.interactions.create(
    model=NEON_MODEL,
    input=(
        "Audit the `WebhookVerifier` implementation and `pytest` suite above "
        "for timing side-channels, replay cache memory growth, or header "
        "parsing edge cases. List at most 3 findings with 1-line fixes."
    ),
    previous_interaction_id=test_turn.id,
    generation_config={
        "thinking_level": get_thinking_level(
            NEON_MODEL, "XHIGH", for_interactions=True
        ),
    },
    store=True,
    stream=True,
    timeout=180,
):
    if event.event_type == "step.delta" and event.delta.type == "text":
        verify_chunks.append(event.delta.text)
    elif (
        event.event_type == "interaction.completed"
        and event.interaction.status != "requires_continuation"
    ):
        verify_turn = event.interaction

stored_interaction_ids.append(verify_turn.id)
print(
    f"Turn 4 [{NEON_MODEL} xhigh stream] "
    f"thought_tokens={verify_turn.usage.total_thought_tokens}, "
    f"output_tokens={verify_turn.usage.total_output_tokens}"
)
display(Markdown("".join(verify_chunks)))

Turn 4 [gemini-4-neon xhigh stream] thought_tokens=8898, output_tokens=455


1. **Non-ASCII & Uppercase `X-Signature-256` Edge Case (Crash / False Reject)**
   `hmac.compare_digest(str, str)` raises an unhandled `TypeError` if `X-Signature-256` contains non-ASCII characters (turning untrusted input into a crash instead of `InvalidSignatureError`) and rejects valid uppercase hex digests against lowercase `.hexdigest()`.
   * **1-line fix:** `if not hmac.compare_digest(expected_signature.encode("ascii"), provided_signature.lower().encode("utf-8")):`

2. **Unbounded Replay Cache Memory Growth & Lock-Scan Timing Side-Channel (`InMemoryReplayStore`)**
   `_seen_event_expirations` has no capacity bound—allowing a burst of valid events within the 300s window to exhaust memory—and runs an $O(N)$ full-dict scan under `self._lock` on every `claim()`, creating lock contention and leaking active cache volume via verification latency.
   * **1-line fix:** Enforce a fail-closed cap before inserting in `claim()` (and use Redis `SET NX EX` in production): `if len(self._seen_event_expirations) >= 10_000: raise WebhookVerificationError("Replay store capacity exceeded.")`

3. **Delimiter Collision in `X-Webhook-Id` (Header Parsing / Replay Bypass)**
   Because `signed_payload` concatenates `f"{webhook_event_id}.{webhook_timestamp}." + payload`, allowing `.` in `X-Webhook-Id` makes the serialization ambiguous: a payload beginning with `<valid_timestamp>.` lets an attacker shift the original timestamp into `X-Webhook-Id` to produce an identical HMAC with a new event ID, bypassing the replay store (and the `pytest` suite currently lacks coverage for malformed headers or deterministic `time.time()` mocking).
   * **1-line fix:** `if not webhook_event_id or "." in webhook_event_id: raise WebhookVerificationError("Missing or invalid X-Webhook-Id header.")`

## Thought signatures and per-model caches on GenerateContent

When managing message history yourself on `GenerateContent`, watch two details:
- In parallel tool-calling turns, only the first `function_call` part carries a `thought_signature` (omitting it in the active turn or duplicating it onto sibling calls returns `HTTP 400`, and stripping earlier signatures degrades reasoning or can trigger `MALFORMED_FUNCTION_CALL`). Append `response.candidates[0].content` as-is and send all `FunctionResponse` parts back in one `role="user"` turn. Switching models across turns is safe (`200 OK`); the server ignores signatures from a different model.
- Implicit 95% KV caches (4K-token minimum) are separate per model. Switching from Neon to Hydrogen misses Hydrogen's cache once on Turn 1, then hits on Turn 2+.

In [15]:
def read_service_file(path: str) -> dict[str, str]:
    """Reads a source file from the repository."""
    files = {
        "gateway/retry.py": (
            "def backoff_ms(attempt: int) -> int:\n"
            "    return 100 * (2 ** attempt)  # missing jitter and max cap\n"
        ),
        "gateway/config.yaml": "\n".join(["max_attempts: 6", "timeout_ms: 2500"]),
    }
    return {"path": path, "content": files.get(path, "FILE_NOT_FOUND")}


def get_endpoint_slo(service: str) -> dict[str, Any]:
    """Fetches p99 latency budget and retry thresholds for a service."""
    return {"service": service, "p99_budget_ms": 1200, "max_backoff_ms": 800}


TOOL_MAP = {
    "read_service_file": read_service_file,
    "get_endpoint_slo": get_endpoint_slo,
}

# Front-load static architecture docs (>4K tokens) for Implicit Caching
SHARED_REPO_CONTEXT = (
    "You are a production reliability assistant. Follow minimal-diff rules.\n\n"
    + "\n".join(
        f"## Module Spec {i:03d}\n"
        f"- Service `svc-{i:03d}` enforces exponential backoff with full jitter, "
        f"bounded `max_backoff_ms`, idempotency keys, and JSON telemetry."
        for i in range(1, 220)
    )
)

### Run parallel tool calls on Neon

In [16]:
agent_config = types.GenerateContentConfig(
    system_instruction=SHARED_REPO_CONTEXT,
    tools=[read_service_file, get_endpoint_slo],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
    thinking_config=types.ThinkingConfig(thinking_level="HIGH"),
)

history: list[types.Content] = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text=(
                    "Call `read_service_file` for `gateway/retry.py` and "
                    "`get_endpoint_slo` for `checkout-gateway` in parallel, "
                    "then diagnose the retry bug."
                )
            )
        ],
    )
]

neon_resp = client.models.generate_content(
    model=NEON_MODEL, contents=history, config=agent_config
)
neon_content = neon_resp.candidates[0].content
history.append(neon_content)

fn_parts = [p for p in neon_content.parts if p.function_call]
print("Neon parallel calls:", [p.function_call.name for p in fn_parts])
print("Has thought_signature:", [bool(p.thought_signature) for p in fn_parts])

tool_response_parts = [
    types.Part.from_function_response(
        name=p.function_call.name,
        response=TOOL_MAP[p.function_call.name](**p.function_call.args),
    )
    for p in fn_parts
]
history.append(types.Content(role="user", parts=tool_response_parts))

Neon parallel calls: ['read_service_file', 'get_endpoint_slo']
Has thought_signature: [True, False]


### Continue the conversation on Hydrogen

Pass `history` (with Neon's `thought_signature` intact) to two back-to-back Hydrogen turns. Turn 1 populates Hydrogen's cache; Turn 2 reuses it.

In [17]:
hydrogen_config = types.GenerateContentConfig(
    system_instruction=SHARED_REPO_CONTEXT,
    tools=[read_service_file, get_endpoint_slo],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
    thinking_config=types.ThinkingConfig(thinking_level="MEDIUM"),
)

# Hydrogen Turn 1: Switch from Neon to Hydrogen (populates Hydrogen's cache)
h1_resp = client.models.generate_content(
    model=HYDROGEN_MODEL, contents=history, config=hydrogen_config
)
history.append(h1_resp.candidates[0].content)
history.append(
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="Write a 3-line patched `backoff_ms` function with full jitter."
            )
        ],
    )
)

# Hydrogen Turn 2: Consecutive turn hits Hydrogen's 95% implicit cache
h2_resp = client.models.generate_content(
    model=HYDROGEN_MODEL, contents=history, config=hydrogen_config
)

for idx, resp in enumerate((h1_resp, h2_resp), start=1):
    u = resp.usage_metadata
    print(
        f"Hydrogen Turn {idx}: prompt_tokens={u.prompt_token_count}, "
        f"cached_tokens={u.cached_content_token_count or 0}, "
        f"thought_tokens={u.thoughts_token_count or 0}"
    )
display(Markdown(h2_resp.text))

Hydrogen Turn 1: prompt_tokens=9582, cached_tokens=0, thought_tokens=94
Hydrogen Turn 2: prompt_tokens=10004, cached_tokens=7533, thought_tokens=137


```python
def backoff_ms(attempt: int) -> int:
    cap = min(800, 100 * (2 ** attempt))
    return random.randint(0, cap)
```
*(Requires `import random` at module scope.)*

## Task router and Planner escalation

An upfront router works well for single-shot tasks (classification, extraction, standalone Q&A) where one model can handle the whole request. For multi-step coding or debugging, start with a Planner + Executor loop and escalate back from Hydrogen to Neon or Carbon when verification fails repeatedly or hits an unsettled design choice.

In [18]:
TASK_ROUTES: dict[str, tuple[str, str]] = {
    "long_horizon_architecture": (CARBON_MODEL, "HIGH"),
    "deep_wide_research": (CARBON_MODEL, "MEDIUM"),
    "frontier_coding": (NEON_MODEL, "HIGH"),
    "secure_code_and_patching": (NEON_MODEL, "HIGH"),
    "science_and_formal_math": (NEON_MODEL, "XHIGH"),
    "worker_subagent_code": (HYDROGEN_MODEL, "HIGH"),
    "fast_tool_turn": (HYDROGEN_MODEL, "MEDIUM"),
    "routing_extraction": (HYDROGEN_MODEL, "LOW"),
}


def route_task(
    task: str, *, for_interactions: bool = False
) -> tuple[str, str]:
    model, raw_level = TASK_ROUTES[task]
    return model, get_thinking_level(
        model, raw_level, for_interactions=for_interactions
    )


def should_escalate_to_planner(
    failed_checks: int, *, unsettled_design: bool = False
) -> tuple[bool, str]:
    if unsettled_design:
        return True, "Escalate to Neon/Carbon: unsettled design choice."
    if failed_checks >= 2:
        return True, f"Escalate to Neon/Carbon: verification failed {failed_checks}x."
    return False, "Keep executing on Hydrogen."


for task_name in TASK_ROUTES:
    mdl, lvl = route_task(task_name)
    print(f"{task_name:<26} -> model={mdl:<18} thinking_level={lvl}")

print()
for label, args in [
    ("First test failure", (1, False)),
    ("Second test failure", (2, False)),
    ("Unsettled API contract", (0, True)),
]:
    escalate, reason = should_escalate_to_planner(args[0], unsettled_design=args[1])
    print(f"{label:<24} -> escalate={str(escalate):<5} | {reason}")

long_horizon_architecture  -> model=gemini-4-carbon    thinking_level=HIGH
deep_wide_research         -> model=gemini-4-carbon    thinking_level=MEDIUM
frontier_coding            -> model=gemini-4-neon      thinking_level=HIGH
secure_code_and_patching   -> model=gemini-4-neon      thinking_level=HIGH
science_and_formal_math    -> model=gemini-4-neon      thinking_level=XHIGH
worker_subagent_code       -> model=gemini-4-hydrogen  thinking_level=HIGH
fast_tool_turn             -> model=gemini-4-hydrogen  thinking_level=MEDIUM
routing_extraction         -> model=gemini-4-hydrogen  thinking_level=LOW

First test failure       -> escalate=False | Keep executing on Hydrogen.
Second test failure      -> escalate=True  | Escalate to Neon/Carbon: verification failed 2x.
Unsettled API contract   -> escalate=True  | Escalate to Neon/Carbon: unsettled design choice.


## Cleaning up

Delete the stored interactions created in the `Interactions API` section in reverse order (last turn first; stored interactions otherwise expire after 7 days).

In [19]:
for interaction_id in reversed(stored_interaction_ids):
    client.interactions.delete(id=interaction_id)

print(f"Deleted {len(stored_interaction_ids)} stored interactions.")

Deleted 4 stored interactions.
